# Stage K Case Study: modifiedTexas2k Grid, Load, Weather, and Decision-Quality Setup

This notebook is a lightweight, first-look case study for the modifiedTexas2k system. It loads the MATPOWER grid, joins bus coordinates, reads the 24-hour load scenarios, visualizes the physical grid, animates the daily load profile, identifies high-load regions, and prepares hooks for weather overlays and contingency/robustness framing.

The intent is exploratory and transparent: understand what the data can support before adapting the existing OPS methodology to a larger Texas-shaped economic case study. This notebook does not call GridFM, GridSFM, Julia, PowerModels, or OPF solvers.

## 1. Setup

The helper module beside this notebook keeps parsing and plotting code out of the main walkthrough. The default paths point to the files currently in `Downloads`; edit `paths` below if you move the data into the repository or an external artifact root.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display

helper_candidates = [
    Path.cwd(),
    Path.cwd() / "experiments" / "test" / "wildfire_tests" / "stage_k_case_study",
]
for helper_dir in helper_candidates:
    if (helper_dir / "stage_k_texas2k_helpers.py").exists():
        sys.path.insert(0, str(helper_dir))
        break

from stage_k_texas2k_helpers import (
    StageKPaths,
    animate_load_24h,
    attach_load_geography,
    build_branch_geography,
    default_paths,
    high_load_tables,
    inspect_pww_coordinate_markers,
    inspect_weather_file,
    load_stage_k_inputs,
    load_weather_overlay_table,
    plot_high_load_regions,
    plot_load_profile,
    plot_load_snapshot,
    plot_physical_grid,
    plot_weather_and_load_overlay,
    plot_weather_variable_overlay,
    plot_weather_variable_panel,
    select_one_day_weather,
    simple_contingency_screen,
    summarize_network,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

In [ ]:
paths = default_paths()

# Example override if the files are moved later:
# paths = StageKPaths(
#     case_file=Path("path/to/modifiedTexas2k.m"),
#     load_file=Path("path/to/modifiedTexas2k_loads_scenarios_0_23.parquet"),
#     coordinate_file=Path("path/to/modifiedTexas2k_coordinates.csv"),
#     weather_file=Path("path/to/weather_overlay.csv"),
# )

for label, path in paths.__dict__.items():
    print(f"{label:16s}: {path}  exists={Path(path).exists() if path else False}")

## 2. Load and Validate the Grid Data

A key mapping detail: MATPOWER branches use `BUS_I` labels such as `1001`, while the coordinate file and load scenario file use zero-based bus row indices. The loader keeps both identities so later methodology code can choose the correct convention explicitly.

Methodology guardrails carried forward from prior stages: preserve canonical bus/branch identities, apply load scenarios through an explicit reference-scenario scaling rule, distinguish visualization diagnostics from optimization/economic claims, and keep weather/hazard overlays as exogenous data until a formal objective is declared.

In [ ]:
case = load_stage_k_inputs(paths)
branch_geo = build_branch_geography(case)
load_geo = attach_load_geography(case, reference_scenario=16)

summary = summarize_network(case, branch_geo)
display(pd.Series(summary, name="modifiedTexas2k summary"))

In [ ]:
bus = case["bus"]
gen = case["gen"]
loads = case["loads"]

checks = {
    "coordinate_rows_equal_bus_rows": len(case["coords"]) == len(bus),
    "load_bus_ids_match_nonzero_case_load_rows": set(loads.loc[loads["scenario"] == 0, "bus_id"].astype(int)) == set(bus.index[bus["PD"] != 0].astype(int)),
    "all_branch_endpoints_have_coordinates": branch_geo[["from_lon", "to_lon", "from_lat", "to_lat"]].notna().all().all(),
    "scenarios": sorted(loads["scenario"].unique().tolist()),
    "rows_per_scenario": loads.groupby("scenario").size().unique().tolist(),
}
display(pd.Series(checks, name="data checks"))

## 3. Physical Grid View

This map is intentionally plain: branches as thin gray lines, load buses in green, and online generator buses in orange. It gives us a clean spatial baseline before layering load, weather, wildfire exposure, de-energization decisions, or contingency results.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
plot_physical_grid(case, branch_geo, ax=ax)
plt.show()

In [ ]:
display(bus["BASE_KV"].value_counts().sort_index().rename("bus_count_by_base_kv").to_frame().T)
display(gen.get("genfuel", pd.Series(dtype=str)).value_counts().rename("generator_count_by_fuel").to_frame().T)

## 4. 24-Hour Load Profile

The scenario table has one row per load bus per scenario. The current case appears calibrated around scenario 16, so `p_case_scaled_mw` preserves the case's per-bus calibration while applying the 24-hour scenario shape.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_load_profile(load_geo, ax=ax, scaled=True)
plt.show()

hourly = load_geo.groupby("scenario")[["p_mw", "q_mvar", "p_case_scaled_mw", "q_case_scaled_mvar"]].sum()
display((hourly / 1000.0).rename(columns=lambda c: c + "_GW_or_GVAr").round(3))

## 5. Load Snapshots: Low, Peak, and Evening Conditions

These snapshots make it easier to see whether load growth is geographically uniform or concentrated in a few regions. The marker size and color both encode active load at the load bus.

In [ ]:
snapshot_scenarios = [0, int(hourly["p_case_scaled_mw"].idxmax()), 20]
fig, axes = plt.subplots(1, len(snapshot_scenarios), figsize=(18, 6), constrained_layout=True)
for ax, scenario in zip(axes, snapshot_scenarios):
    plot_load_snapshot(case, branch_geo, load_geo, scenario=scenario, ax=ax, title=f"Scenario {scenario}")
plt.show()

## 6. Animation: 24-Hour Load Changes

Run the next cell to render an inline animation. If the notebook frontend does not display the animation, rerun the cell after restarting the kernel; it uses Matplotlib's built-in JavaScript HTML representation and does not need ffmpeg.

In [ ]:
anim = animate_load_24h(case, branch_geo, load_geo, interval_ms=550)
HTML(anim.to_jshtml())

## 7. Identify High-Load Buses and Regions

The bus table ranks individual load buses by peak scaled load. The region table bins the map into coarse geographic cells; this is a first-look way to find demand centers before introducing economic value of lost load, wildfire exposure, or contingency severity.

In [ ]:
top_load_buses, high_load_regions = high_load_tables(load_geo, top_n=25, region_bins=8)

display(top_load_buses[["bus_id", "BUS_I", "bus_name", "lon", "lat", "avg_p_mw", "peak_p_mw", "min_p_mw", "peak_scenario"]].round(3))
display(high_load_regions.head(15).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
plot_high_load_regions(case, branch_geo, high_load_regions, ax=ax, top_n=14)
plt.show()

## 8. Weather / Heat-Risk Overlay Hook

The load data covers one 24-hour block. The `.pww` file may contain multiple days of weather, so the workflow below selects one 24-step weather snippet and aligns `weather_step = 0..23` to load `scenario = 0..23`.

A `.pww` file is present in Downloads, but this file identifies as a binary PowerWorld timestep weather file. The notebook can inspect visible coordinate markers, but it does not decode undocumented binary weather values. For actual overlays, export a one-day or multi-day weather table from PowerWorld to CSV/Parquet first.

Expected overlay columns are flexible: `lon`/`longitude`/`x`, `lat`/`latitude`/`y`, optional `timestamp`/`datetime`/`time`/`hour`, and weather columns for `temperature`, `dew_point`, and `wind_100m`. Common aliases such as `temp_f`, `dewpoint`, `wind_speed_100m`, and `ws100` are accepted.

In [ ]:
weather_info = inspect_weather_file(paths.weather_file)
display(pd.Series(weather_info, name="weather file inspection"))

if weather_info.get("exists") and str(paths.weather_file).lower().endswith(".pww"):
    pww_markers = inspect_pww_coordinate_markers(paths.weather_file)
    display(pd.Series({
        "visible_coordinate_markers": len(pww_markers),
        "marker_lon_bounds": (pww_markers["lon"].min(), pww_markers["lon"].max()) if len(pww_markers) else None,
        "marker_lat_bounds": (pww_markers["lat"].min(), pww_markers["lat"].max()) if len(pww_markers) else None,
    }, name="pww coordinate marker scan"))
    display(pww_markers.head())

### PowerWorld Export Target

Use PowerWorld to open/decode the `.pww` file, then export the weather grid or bus-sampled weather to CSV or Parquet. For this notebook, the cleanest export is one row per weather point per timestamp:

| lon | lat | timestamp | temperature | dew_point | wind_100m |
| ---: | ---: | --- | ---: | ---: | ---: |
| -99.75 | 31.25 | 2023-06-25 16:00 | 103.2 | 64.5 | 12.1 |

A multi-day export is acceptable. Set `WEATHER_DAY` below to the desired date, and the notebook will take the first 24 timestamps for that day.

In [ ]:
# Preferred real export target from PowerWorld. A multi-day export is fine;
# select_one_day_weather will take a 24-step snippet.
# Preferred export columns: lon, lat, timestamp, temperature, dew_point, wind_100m.
real_weather_export_candidates = [
    Path("data/stage_k_powerworld_weather_export.parquet"),
    Path("data/stage_k_powerworld_weather_export.csv"),
    Path.cwd() / "experiments" / "test" / "wildfire_tests" / "stage_k_case_study" / "data" / "stage_k_powerworld_weather_export.parquet",
    Path.cwd() / "experiments" / "test" / "wildfire_tests" / "stage_k_case_study" / "data" / "stage_k_powerworld_weather_export.csv",
]
raw_powerworld_csv_candidates = [
    Path("data/stage_k_powerworld_weather_input_raw.csv"),
    Path.cwd() / "experiments" / "test" / "wildfire_tests" / "stage_k_case_study" / "data" / "stage_k_powerworld_weather_input_raw.csv",
]
preview_weather_candidates = [
    Path("data/stage_k_weather_preview_texas_2023_06_25.parquet"),
    Path.cwd() / "experiments" / "test" / "wildfire_tests" / "stage_k_case_study" / "data" / "stage_k_weather_preview_texas_2023_06_25.parquet",
]

weather_overlay_path = next((p for p in real_weather_export_candidates if p.exists()), None)
weather_source_kind = "powerworld_export"

if weather_overlay_path is None:
    raw_powerworld_csv = next((p for p in raw_powerworld_csv_candidates if p.exists()), None)
    if raw_powerworld_csv is not None:
        from convert_powerworld_weather_export import normalize_powerworld_weather

        export_path = raw_powerworld_csv.with_name("stage_k_powerworld_weather_export.parquet")
        normalize_powerworld_weather(raw_powerworld_csv, export_path)
        weather_overlay_path = export_path

if weather_overlay_path is None:
    weather_overlay_path = next((p for p in preview_weather_candidates if p.exists()), paths.weather_file)
    weather_source_kind = "modeled_preview"

print(f"Weather table selected: {weather_overlay_path}")
print(f"Weather source kind: {weather_source_kind}")
WEATHER_DAY = None  # Example: "2023-06-25" if the export has datetime/date values.
WEATHER_SCENARIO_TO_PLOT = 16

try:
    weather_raw = load_weather_overlay_table(weather_overlay_path)
    weather = select_one_day_weather(weather_raw, day=WEATHER_DAY, max_steps=24)
    if weather_source_kind == "modeled_preview":
        print("WARNING: displaying modeled preview weather values, not decoded PowerWorld .pww values.")
        print("Replace data/stage_k_powerworld_weather_export.parquet with a PowerWorld export for real weather plots.")
    display(weather.groupby("scenario").size().rename("weather_points_per_aligned_scenario").to_frame().T)
    display(weather.head())

    plot_weather_variable_panel(
        case,
        branch_geo,
        weather,
        scenario=WEATHER_SCENARIO_TO_PLOT,
        variables=("temperature", "dew_point", "wind_100m"),
    )
    plt.show()
except Exception as exc:
    print("Weather overlay not rendered yet.")
    print(exc)
    print("Export a one-day or multi-day weather/heat-risk table from the .pww file to CSV/Parquet, then rerun this cell.")
    print("Preferred columns: lon, lat, timestamp, temperature, dew_point, wind_100m.")

### Larger Separate Weather Graphs

After the PowerWorld export is available, the next cell renders temperature, dew point, and 100m wind as separate full-size grid overlays. Keeping the variables separate makes it easier to see whether different weather drivers stress different parts of the grid.

In [ ]:
if "weather" in globals():
    weather_plot_specs = [
        ("temperature", "Temperature"),
        ("dew_point", "Dew Point"),
        ("wind_100m", "100m Wind Speed"),
    ]
    for variable, label in weather_plot_specs:
        if variable in weather.columns:
            fig, ax = plt.subplots(figsize=(11, 9))
            plot_weather_variable_overlay(
                case,
                branch_geo,
                weather,
                variable=variable,
                scenario=WEATHER_SCENARIO_TO_PLOT,
                ax=ax,
                label=label,
            )
            plt.show()
        else:
            print(f"Skipping {label}: column {variable!r} not found in weather export.")
else:
    print("Run the PowerWorld weather-export overlay cell above after exporting the .pww data to CSV/Parquet.")

## 9. First-Look Contingency Screen

This is not an OPS result. It is a transparent first-look screen using only observable branch attributes: geographically long lines and low-rated lines receive higher attention. Once weather/hazard values are joined to branches, this table can be replaced by a wildfire exposure, service-impact, and exact-contingency score.

In [ ]:
contingency_candidates = simple_contingency_screen(branch_geo, top_n=25)
display(contingency_candidates.round(3))

## 10. How This Connects to Stage K Methodology

A practical Stage K progression can now be built from this notebook:

1. Physical/economic baseline: solve intact AC OPF for selected hours, record cost, generation mix, voltage/thermal margins, and load pockets.
2. Weather/hazard join: convert heat/wildfire/weather data into bus or branch exposure scores, then visualize exposure against high-load regions and long corridors.
3. Contingency screen: rank lines using exposure, intact loading, branch rating, geographic length, and service-impact proxies.
4. Decision-quality study: compare candidate de-energization/load-service decisions under DC, future surrogate evaluation, and exact fixed-decision AC audits in a separate execution notebook.
5. Robustness study: test whether selected decisions remain acceptable across the 24-hour load profile and plausible contingency/weather perturbations.

The notebook deliberately separates data understanding from optimization claims. The maps and tables here are the shared vocabulary for the economic case study and contingency analysis that can follow.